In [1]:
%load_ext autoreload
%autoreload 2
import sys
import pandas as pd
import os
import matplotlib.pyplot as plt
import cortex
import seaborn as sns
from os.path import join
from collections import defaultdict
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib
import dvu
from neuro.flatmaps_helper import load_flatmaps
from neuro.features.questions.gpt4 import QS_35_STABLE
import sys
import warnings
sys.path.append('../notebooks')
from tqdm import tqdm
from neuro import config
from neuro import analyze_helper
import neuro.viz
from neuro.features.qa_questions import get_questions, get_merged_questions_v3_boostexamples
# flatmaps_per_question = __import__('06_flatmaps_per_question')
import viz
import gct
from neuro.flatmaps_helper import load_flatmaps
from statsmodels.stats.multitest import multipletests

/home/chansingh/automated-brain-explanations/.venv/lib/python3.12/site-packages/spacy/cli/_util.py:23: DeprecationWarning: Importing 'parser.split_arg_string' is deprecated, it will only be available in 'shell_completion' in Click 9.0.
  from click.parser import split_arg_string
/home/chansingh/automated-brain-explanations/.venv/lib/python3.12/site-packages/weasel/util/config.py:8: DeprecationWarning: Importing 'parser.split_arg_string' is deprecated, it will only be available in 'shell_completion' in Click 9.0.
  from click.parser import split_arg_string


Note, this notebook requires first running `03_export_qa_flatmaps.ipynb` into `df_qa_dict.pkl` files for each subject.

In [2]:
qa_questions_list = QS_35_STABLE
subject = 'S02'

In [6]:
# load qa weights
corrs_df_dict = {}
frac_voxels_to_keep_list = [0.01, 0.05, 0.1, 0.25, 0.5, 1]

# corrs used for masking
corrs_test = joblib.load(join(config.PROCESSED_DIR, subject.replace(
    'UT', ''), 'corrs_test_35.pkl')).values[0]
corrs_test_individual_dict = joblib.load(join(config.PROCESSED_DIR, subject.replace(
    'UT', ''), 'corrs_test_individual_gpt4_qs_35.pkl'))

In [7]:
def mask_voxels(df, frac_voxels_to_keep, corrs_mask_per_question=True):
    df = df.copy()
    if frac_voxels_to_keep < 1:
        # mask based on corrs
        if corrs_mask_per_question:
            for i in range(df.shape[0]):
                q = df.index[i]
            # q = questions_names_df['qa'].values[i]
                mask = (corrs_test_individual_dict[q] > np.percentile(
                    corrs_test_individual_dict[q], 100 * (1 - frac_voxels_to_keep))).astype(bool)
                for col in range(len(df.columns)):
                    df.iloc[i, col] = df.iloc[i, col][mask]
        else:
            mask = (corrs_test > np.percentile(
                corrs_test, 100 * (1 - frac_voxels_to_keep))).astype(bool)

            for i in range(df.shape[0]):
                for col in range(len(df.columns)):
                    df.iloc[i, col] = df.iloc[i, col][mask]
    return df

def compute_corrs_with_first_col(df):
    correlations_to_first = defaultdict(list)
    for col_idx in range(1, df.shape[1]):
        corrs = []
        for i in range(df.shape[0]):
            corrs.append(np.corrcoef(
                df.iloc[i, 0],
                df.iloc[i, col_idx]
            )[0, 1])
        correlations_to_first[col_idx] = corrs
    d = pd.DataFrame(correlations_to_first, index=QS_35_STABLE)
    d.columns = df.columns[1:]
    return d

In [8]:
settings = [
    'individual_gpt4',
    'full_35_gpt4_pc',
    'individual_gpt4_pc_new',
    'individual_gpt4_wordrate', 
]
frac_voxels_to_keep = 0.5

flatmap_lists_by_setting = defaultdict(list)
for setting in tqdm(settings): 
    # setting = 'individual_gpt4_pc_new'
    flatmaps_qa_dict = joblib.load(
        join(config.PROCESSED_DIR, subject.replace('UT', ''), setting + '.pkl'))
    for q in QS_35_STABLE:
        flatmap = flatmaps_qa_dict[q]
        flatmap_lists_by_setting[setting].append(flatmaps_qa_dict[q])


df_flatmap_by_setting_main = pd.DataFrame(flatmap_lists_by_setting, index=QS_35_STABLE)
df_flatmap_by_setting_25 = mask_voxels(
    df_flatmap_by_setting_main, frac_voxels_to_keep=0.25, corrs_mask_per_question=True)
df_flatmap_by_setting_1 = mask_voxels(
    df_flatmap_by_setting_main, frac_voxels_to_keep=0.1, corrs_mask_per_question=True)
df_flatmap_by_setting_01 = mask_voxels(
    df_flatmap_by_setting_main, frac_voxels_to_keep=0.01, corrs_mask_per_question=True)


# compute the correlation between flatmaps of every column to the first column
d = compute_corrs_with_first_col(df_flatmap_by_setting_main)
d_25 = compute_corrs_with_first_col(df_flatmap_by_setting_25)
d_1 = compute_corrs_with_first_col(df_flatmap_by_setting_1)
d_01 = compute_corrs_with_first_col(df_flatmap_by_setting_01)


d_full = pd.concat(
    (
    pd.DataFrame(d.mean(axis=0)).T,
    pd.DataFrame(d_25.mean(axis=0)).T,
    pd.DataFrame(d_1.mean(axis=0)).T,
    pd.DataFrame(d_01.mean(axis=0)).T,
    d_1.sort_values(by=d_1.columns[-1], ascending=False)
    ),
    ignore_index=False
).round(3)
d_full.index = [
    'AVG (all voxels)',
    'AVG (25% top-predicted)',
    'AVG (10% top-predicted)',
    'AVG (1% top-predicted)'
] + list(d.index)
d_full = d_full.rename(columns={
    'full_35_gpt4_pc': '+Response PCs, +Joint fitting',
    'individual_gpt4_pc_new': '+Response PCs',
    'individual_gpt4_wordrate': '+Word rate',
})
# reverse col order
d_full = d_full[d_full.columns[::-1]]
# d_full.to_latex()
d_full

  0%|          | 0/4 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 36.67it/s]


,+Word rate,+Response PCs,"+Response PCs, +Joint fitting"
AVG (all voxels),0.663,0.895,0.736
AVG (25% top-predicted),0.734,0.933,0.814
AVG (10% top-predicted),0.765,0.946,0.838
AVG (1% top-predicted),0.802,0.950,0.842
Does the sentence describe a personal reflection or thought?,0.956,0.991,0.933
Does the sentence contain a proper noun?,0.938,0.971,0.833
Does the sentence describe a physical action?,0.929,0.982,0.876
Does the sentence describe a personal or social interaction that leads to a change or revelation?,0.916,0.995,0.877
Does the sentence involve the mention of a specific object or item?,0.900,0.973,0.946
Does the sentence involve a description of physical environment or setting?,0.887,0.985,0.819


In [9]:
d_full.index = d_full.index.map(neuro.analyze_helper.abbrev_question)
print(d_full.style.format(precision=3).to_latex(hrules=True).replace('%', '\%'))

\begin{tabular}{lrrr}
\toprule
 & +Word rate & +Response PCs & +Response PCs, +Joint fitting \\
\midrule
AVG (all voxels) & 0.663 & 0.895 & 0.736 \\
AVG (25\% top-predicted) & 0.734 & 0.933 & 0.814 \\
AVG (10\% top-predicted) & 0.765 & 0.946 & 0.838 \\
AVG (1\% top-predicted) & 0.802 & 0.950 & 0.842 \\
...describe a personal reflection or thought? & 0.956 & 0.991 & 0.933 \\
...contain a proper noun? & 0.938 & 0.971 & 0.833 \\
...describe a physical action? & 0.929 & 0.982 & 0.876 \\
...describe a personal or social interaction? & 0.916 & 0.995 & 0.877 \\
...mention a specific object or item? & 0.900 & 0.973 & 0.946 \\
...describe a physical environment or setting? & 0.887 & 0.985 & 0.819 \\
...describe a relationship between people? & 0.882 & 0.989 & 0.921 \\
...mention a specific location? & 0.878 & 0.993 & 0.931 \\
...mention time? & 0.877 & 0.800 & 0.892 \\
...abstract rather than concrete? & 0.872 & 0.994 & 0.886 \\
...express an opinion about an event or character? & 0.872 & 0.978

<>:2: SyntaxWarning: invalid escape sequence '\%'
<>:2: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_2088400/2038805771.py:2: SyntaxWarning: invalid escape sequence '\%'
  print(d_full.style.format(precision=3).to_latex(hrules=True).replace('%', '\%'))


In [19]:
# q = 'Does the input describe a specific texture or sensation?'
q = 'Does the sentence mention a specific location?'
flatmaps_location = df_flatmap_by_setting_main.loc[q]

for k in flatmaps_location.index:
    print(k)
    neuro.viz.quickshow(
        flatmaps_location[k].flatten(),
        fname_save=join('compare_variations', f'{q[-10:-1].strip()}_{k.replace(" ", "_")}_{subject}.png'),
        kwargs={'with_rois': False},
        with_colorbar=False,)

individual_gpt4
full_35_gpt4_pc
individual_gpt4_pc_new
individual_gpt4_wordrate
